# Day 8 v2 — Silver Transformation: All 17 Entities with For Loop
**Same logic as v1, extended to all entities using a for loop**

**Source:** `/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/api/<entity>/`
**Sink:** `/Volumes/dbw_ev_intelligence_dev/default/silver-volume/api/<entity>/` (Delta)

**What changed from v1:**
- Added an entity config list (name, natural key, column cast map)
- Wrapped the same v1 steps inside a `for` loop over all 17 entities
- Each entity goes through: read -> explode -> cast -> audit cols -> deduplicate -> write
- One summary printed at the end

**What stays the same as v1:**
- Every transformation step is identical
- Full overwrite mode (no merge yet — that is v3)
- Same Silver audit columns

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.types import MapType, StringType
from pyspark.sql.window import Window
from datetime import datetime, timezone

print("Imports OK")

In [ ]:
# Cell 2 — Constants
BRONZE_BASE = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/api"
SILVER_BASE = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/api"

RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"Bronze base : {BRONZE_BASE}")
print(f"Silver base : {SILVER_BASE}")
print(f"Run time    : {RUN_TS}")

In [ ]:
# Cell 3 — Entity config list
# Each dict has:
#   entity_name : folder name in Bronze/Silver
#   natural_key : the unique ID column for deduplication
#   cast        : dict of {column_name: spark_type} for type casting
#
# Compare to v1 where we hardcoded the payments columns directly.
# Here we store the config for all 17 entities and loop over them.

ENTITIES = [
    {
        "entity_name": "payments",
        "natural_key": "payment_id",
        "cast": {
            "payment_id": "string", "session_id": "string", "customer_id": "string",
            "amount_aud": "decimal(10,2)", "currency": "string", "status": "string",
            "payment_method": "string", "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    {
        "entity_name": "sessions",
        "natural_key": "session_id",
        "cast": {
            "session_id": "string", "vehicle_id": "string", "charger_id": "string",
            "customer_id": "string", "station_id": "string",
            "start_time": "timestamp", "end_time": "timestamp",
            "energy_kwh": "decimal(10,4)", "duration_minutes": "integer",
            "status": "string", "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    {
        "entity_name": "customers",
        "natural_key": "customer_id",
        "cast": {
            "customer_id": "string", "first_name": "string", "last_name": "string",
            "email": "string", "phone": "string", "city": "string",
            "state": "string", "country": "string",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    {
        "entity_name": "fleet",
        "natural_key": "vehicle_id",
        "cast": {
            "vehicle_id": "string", "make": "string", "model": "string",
            "year": "integer", "battery_capacity": "decimal(8,2)", "range_km": "decimal(8,2)",
            "status": "string", "partner_id": "string",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    {
        "entity_name": "chargers",
        "natural_key": "charger_id",
        "cast": {
            "charger_id": "string", "station_id": "string", "charger_type": "string",
            "power_kw": "decimal(8,2)", "status": "string", "connector_type": "string",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    {
        "entity_name": "vehicles",
        "natural_key": "vehicle_id",
        "cast": {
            "vehicle_id": "string", "customer_id": "string", "make": "string",
            "model": "string", "year": "integer",
            "battery_capacity": "decimal(8,2)", "range_km": "decimal(8,2)",
            "license_plate": "string", "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    {
        "entity_name": "stations",
        "natural_key": "station_id",
        "cast": {
            "station_id": "string", "name": "string", "city": "string",
            "state": "string", "country": "string",
            "latitude": "decimal(10,7)", "longitude": "decimal(10,7)",
            "total_chargers": "integer", "status": "string",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    {
        "entity_name": "complaints",
        "natural_key": "complaint_id",
        "cast": {
            "complaint_id": "string", "customer_id": "string", "session_id": "string",
            "category": "string", "description": "string", "status": "string",
            "priority": "string", "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    {
        "entity_name": "maintenance_events",
        "natural_key": "event_id",
        "cast": {
            "event_id": "string", "charger_id": "string", "station_id": "string",
            "event_type": "string", "description": "string", "technician_id": "string",
            "status": "string", "scheduled_date": "date", "completed_date": "date",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    {
        "entity_name": "energy_prices",
        "natural_key": "price_id",
        "cast": {
            "price_id": "string", "region": "string",
            "price_per_kwh": "decimal(8,4)", "currency": "string",
            "effective_from": "timestamp", "effective_to": "timestamp",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    {
        "entity_name": "tariffs",
        "natural_key": "tariff_id",
        "cast": {
            "tariff_id": "string", "name": "string",
            "price_per_kwh": "decimal(8,4)", "price_per_min": "decimal(8,4)",
            "currency": "string", "valid_from": "timestamp", "valid_to": "timestamp",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    {
        "entity_name": "charge_cards",
        "natural_key": "card_id",
        "cast": {
            "card_id": "string", "customer_id": "string", "card_number": "string",
            "status": "string", "issued_at": "timestamp", "expires_at": "timestamp",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    {
        "entity_name": "employees",
        "natural_key": "employee_id",
        "cast": {
            "employee_id": "string", "first_name": "string", "last_name": "string",
            "email": "string", "role": "string", "department": "string",
            "station_id": "string", "hire_date": "date",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    {
        "entity_name": "partners",
        "natural_key": "partner_id",
        "cast": {
            "partner_id": "string", "name": "string", "country": "string",
            "contact_email": "string", "status": "string",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    {
        "entity_name": "cities",
        "natural_key": "city_id",
        "cast": {
            "city_id": "string", "name": "string", "state_code": "string",
            "country": "string", "latitude": "decimal(10,7)", "longitude": "decimal(10,7)",
            "population": "long", "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    {
        "entity_name": "states",
        "natural_key": "state_code",
        "cast": {
            "state_code": "string", "name": "string", "country": "string",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
    {
        "entity_name": "weather",
        "natural_key": "city_id",
        "cast": {
            "city_id": "string", "temperature_c": "decimal(5,2)",
            "humidity_pct": "decimal(5,2)", "wind_speed_kmh": "decimal(6,2)",
            "condition": "string", "recorded_at": "timestamp",
            "created_at": "timestamp", "updated_at": "timestamp",
        }
    },
]

print(f"Entities configured: {len(ENTITIES)}")
for e in ENTITIES:
    print(f"  {e['entity_name']:<25} natural_key={e['natural_key']}")

In [ ]:
# Cell 4 — For loop: transform all 17 entities
# This is the same logic as v1 (payments), now applied to every entity.
# Each iteration:
#   1. Read Bronze JSON
#   2. Explode data[]  (handles array<string> edge case for small entities)
#   3. Cast columns
#   4. Add audit columns
#   5. Deduplicate on natural key
#   6. Write to Silver Delta (overwrite)

results = []

for entity in ENTITIES:
    entity_name = entity["entity_name"]
    natural_key = entity["natural_key"]
    cast_map    = entity["cast"]

    bronze_path = f"{BRONZE_BASE}/{entity_name}"
    silver_path = f"{SILVER_BASE}/{entity_name}"

    print(f"Processing: {entity_name} ...", end=" ")

    try:
        # Step 1: Read
        raw_df = spark.read.option("multiline", "true").json(bronze_path)

        # Step 2: Explode data[] — handle array<string> for small entities
        exploded_df = raw_df.select(F.explode(F.col("data")).alias("r"))
        if dict(exploded_df.dtypes)["r"] == "string":
            flat_df = (
                exploded_df
                .select(F.from_json(F.col("r"), MapType(StringType(), StringType())).alias("r"))
                .select("r.*")
            )
        else:
            flat_df = exploded_df.select("r.*")

        bronze_rows = flat_df.count()

        # Step 3: Cast columns
        cast_exprs = [
            F.col(c).cast(t).alias(c)
            for c, t in cast_map.items()
            if c in flat_df.columns
        ]
        typed_df = flat_df.select(cast_exprs)

        # Step 4: Add Silver audit columns
        audited_df = (
            typed_df
            .withColumn("silver_ingested_at", F.lit(RUN_TS).cast("timestamp"))
            .withColumn("silver_load_type",   F.lit("full"))
            .withColumn("silver_pipeline",    F.lit("pl_silver_all_entities_v2"))
        )

        # Step 5: Deduplicate — keep latest updated_at per natural key
        window = Window.partitionBy(natural_key).orderBy(F.col("updated_at").desc())
        deduped_df = (
            audited_df
            .withColumn("_row_num", F.row_number().over(window))
            .filter(F.col("_row_num") == 1)
            .drop("_row_num")
        )
        silver_rows = deduped_df.count()

        # Step 6: Write to Silver Delta
        (
            deduped_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .save(silver_path)
        )

        print(f"OK  (bronze={bronze_rows} -> silver={silver_rows})")
        results.append({"entity": entity_name, "status": "succeeded",
                        "bronze_rows": bronze_rows, "silver_rows": silver_rows, "error": None})

    except Exception as e:
        print(f"FAILED")
        results.append({"entity": entity_name, "status": "failed",
                        "bronze_rows": 0, "silver_rows": 0, "error": str(e)})

print("\nAll entities done.")

In [ ]:
# Cell 5 — Run summary

succeeded = [r for r in results if r["status"] == "succeeded"]
failed    = [r for r in results if r["status"] == "failed"]

print("=" * 65)
print("SILVER TRANSFORMATION v2 — RUN SUMMARY")
print("=" * 65)
print(f"  run_timestamp  : {RUN_TS}")
print(f"  entities total : {len(results)}")
print(f"  succeeded      : {len(succeeded)}")
print(f"  failed         : {len(failed)}")
print()

for r in results:
    tag = "[OK]  " if r["status"] == "succeeded" else "[FAIL]"
    print(f"  {tag} {r['entity']:<25} bronze={r['bronze_rows']:>6}  silver={r['silver_rows']:>6}")
    if r["error"]:
        print(f"         Error: {r['error']}")

if failed:
    raise Exception(f"{len(failed)} entity(ies) failed — check output above.")
else:
    print(f"\nAll {len(succeeded)} entities written to Silver successfully.")